In [9]:
import os
import glob
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

In [ ]:
DATA_PATH = "/home/justine/code/Maelle05/DyslexIA/data/data_T4/data"
files_raw = glob.glob(os.path.join(DATA_PATH, "*raw.csv"))

LABELS_PATH = "/home/justine/code/Maelle05/DyslexIA/data/dyslexia_class_label.csv"

TARGET_LEN = 2000 # A FINE TUNER
GAZE_COLS  = ['gaze_x_left', 'gaze_y_left', 'gaze_x_right', 'gaze_y_right']
SID_COL    = 'subject_id'

In [6]:
labels = pd.read_csv(LABELS_PATH)
print(f"\nDistribution : {labels['class_id'].value_counts().to_dict()}")


Distribution : {0: 35, 1: 35}


In [7]:
all_lengths = []

for f in files_raw:
    df = pd.read_csv(f)
    all_lengths.append(len(df))

print("min:", min(all_lengths))
print("max:", max(all_lengths))
print("mean:", sum(all_lengths)/len(all_lengths))

min: 11511
max: 100912
mean: 27784.114285714284


In [ ]:
def to_fixed_length(signal, target_len):
    """Interpolation linéaire vers une longueur fixe."""
    x_old = np.linspace(0, 1, len(signal))
    x_new = np.linspace(0, 1, target_len)
    return interp1d(x_old, signal, kind='linear')(x_new)

In [ ]:
def extract_features_selected(df, gaze_cols, target_len):
    signals = {}
    for col in gaze_cols:
        s = to_fixed_length(df[col].values.astype(float), target_len)
        signals[col] = s

    def vel_percentile75(s):
        return np.percentile(np.abs(np.diff(s)), 75)

    def vel_saccade_rate(s):
        vel = np.abs(np.diff(s))
        vel_mean = np.mean(vel)
        vel_std  = np.std(vel) + 1e-8
        return np.sum(vel > vel_mean + 2 * vel_std) / len(vel)

    def vel_std_fixation_proxy(s):
        vel = np.abs(np.diff(s))
        slow_mask = vel < np.percentile(vel, 25)
        runs = np.diff(np.concatenate([[0], slow_mask.astype(int), [0]]))
        run_lengths = np.where(runs == -1)[0] - np.where(runs == 1)[0]
        return np.std(run_lengths) if len(run_lengths) > 0 else 0.0

    def vel_median(s):
        return np.median(np.abs(np.diff(s)))

    def temporal_p90(s):
        return np.percentile(s, 90)

    feat = np.array([
        vel_percentile75(signals['gaze_x_left']),
        vel_saccade_rate(signals['gaze_x_left']),
        vel_std_fixation_proxy(signals['gaze_x_left']),
        vel_std_fixation_proxy(signals['gaze_y_left']),
        temporal_p90(signals['gaze_y_left']),
        vel_median(signals['gaze_x_right']),
        vel_percentile75(signals['gaze_x_right']),
        vel_std_fixation_proxy(signals['gaze_x_right']),
    ])

    return np.nan_to_num(feat, nan=0.0)